<a href="https://colab.research.google.com/github/Jchang118/d2l/blob/main/ndarray.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 数据操作

## 入门

1. 使用`arange`创建一个行向量`x`

In [ ]:
import torch

In [ ]:
x = torch.arange(12)
x

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

2. 通过张量的`shape`属性来访问张量(沿每个轴的长度)的形状

In [ ]:
x.shape

torch.Size([12])

3. 通过张量的`numel`函数来检查它的大小(size)

In [ ]:
x.numel()

12

4. 通过张量的`reshape`函数来改变其形状;我们也可以使用`x.reshape(-1,4)`或`x.reshape(3,-1)`来取代`x.reshape(3,4)`

In [ ]:
X = x.reshape(3, 4)
X

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

5. 全0全1初始化矩阵

In [ ]:
torch.zeros((2, 3, 4))

tensor([[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]])

In [ ]:
torch.ones((2, 3, 4))

tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])

6. 通过某个特定的概率分布中随机采样来得到张量中每个元素的值,例如,创建一个形状为(3,4)的张量,其中的每个元素都从均值为0、标准差为1的标准高斯分布(正态分布)中随机采样

In [ ]:
torch.randn(3, 4)

tensor([[ 6.5486e-01,  6.0536e-01,  7.4836e-01,  5.2399e-02],
        [ 2.3199e-01, -9.8647e-01, -1.4142e+00, -1.0452e+00],
        [ 3.2041e-01, -1.6427e-04, -1.2988e-01,  3.9900e-01]])

7. 通过提供包含数值的Python列表(或嵌套列表),来为所需张量中的每个元素赋予确定值

In [ ]:
torch.tensor([[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

tensor([[2, 1, 4, 3],
        [1, 2, 3, 4],
        [4, 3, 2, 1]])

### 运算符

我们的兴趣不仅限于读取数据和写入数据.我们想在这些数据上执行数学运算,其中最简单且最有用的操作是按元素(elementwise)运算.对于任意具有相同形状的张量,**[常见的标准算术运算符(`+`、`-`、`*`、`/`和`**`)都可以被升级为按元素运算]**.我们可以在同一形状的任意两个张量上调用按元素操作.

In [10]:
x = torch.tensor([1.0, 2, 4, 8])
y = torch.tensor([2, 2, 2, 2])
x + y, x - y, x * y, x / y, x ** y

(tensor([ 3.,  4.,  6., 10.]),
 tensor([-1.,  0.,  2.,  6.]),
 tensor([ 2.,  4.,  8., 16.]),
 tensor([0.5000, 1.0000, 2.0000, 4.0000]),
 tensor([ 1.,  4., 16., 64.]))

(**“按元素”方式可以应用更多的计算**),包括像求幂这样的一元运算符.

In [13]:
torch.exp(x)

tensor([2.7183e+00, 7.3891e+00, 5.4598e+01, 2.9810e+03])

除了按元素计算外,我们还可以执行线性代数运算,包括向量点积和矩阵乘法.我们将在`sec_linear-algebra`中解释线性代数的重点内容.

[**我们也可以把多个张量连结(concatenate)在一起**],把它们端对端地叠起来形成一个更大的张量.注意:`dim=0`沿行连结,`dim=1`沿列连结.

In [14]:
X = torch.arange(12, dtype=torch.float32).reshape((3, 4))
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
torch.cat((X, Y), dim=0), torch.cat((X, Y), dim=1)

(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.],
         [ 2.,  1.,  4.,  3.],
         [ 1.,  2.,  3.,  4.],
         [ 4.,  3.,  2.,  1.]]),
 tensor([[ 0.,  1.,  2.,  3.,  2.,  1.,  4.,  3.],
         [ 4.,  5.,  6.,  7.,  1.,  2.,  3.,  4.],
         [ 8.,  9., 10., 11.,  4.,  3.,  2.,  1.]]))

[**通过逻辑运算符构建二元张量**]

In [15]:
X == Y

tensor([[False,  True, False,  True],
        [False, False, False, False],
        [False, False, False, False]])

[**对张量中的所有元素进行求和,会产生一个单元素张量.**]

In [16]:
X.sum()

tensor(66.)

### 广播机制

在上面的部分中,我们看到了如何在相同形状的两个张量上执行按元素操作.在某些情况下,[**即使形状不同我们然然可以通过调用*广播机制*(broardcasting mechanisim)来执行按元素操作**]. 这种机制的工作方式如下:
1. 通过适当复制元素来扩展一个或两个数组,以便在转换之后,两个张量具有相同的形状j;
2. 对生成的数组执行按元素操作.

在大多数情况下,我们将按着数组中长度为1的轴进行广播,如下例子:



In [17]:
a = torch.arange(3).reshape((3, 1))
b = torch.arange(2).reshape((1, 2))
a, b

(tensor([[0],
         [1],
         [2]]),
 tensor([[0, 1]]))

由于`a`和`b`分别是3 x 1和1 x 2矩阵,如果让它们相加,它们的形状不匹配.我们将两个矩阵*广播*为一个更大的3 x 2矩阵:

In [18]:
a + b

tensor([[0, 1],
        [1, 2],
        [2, 3]])